In [5]:
import os
import imageio
from numba import jit
from matplotlib import pyplot as plt
import numpy as np

os.makedirs('images', exist_ok=True)

N0 = 1000
N = 5000
Ls = [500]
J = 1.
T = 2.
Ns = [1, 10, 100, 1000, 5000]
filenames = []
r_0 = np.arange(1, Ls[0] / 2, 1)
r_0 = [ x for r_0 in ]

print(r_0)


def hihi(current_lattice, L_val, r_0):
    hi = 0
    for i in range(L_val):
        for j in range(L_val):
            hi += current_lattice[i][j] * current_lattice[i][(i+r_0) % L_val]
    hi /= L_val**2
    return hi



@jit(nopython = True)
def vonNeumanNeighborhood(x, y, L_val):
    neighborurs = []
    neighborurs.append([(x + 1) % L_val, y])
    neighborurs.append([(x - 1) % L_val, y])
    neighborurs.append([x, (y + 1) % L_val])
    neighborurs.append([x, (y - 1) % L_val])
    return np.array(neighborurs)


@jit(nopython = True)
def mcs(T, L_val, current_lattice):
    for _ in range(L_val**2):
        i = np.array([np.random.randint(L_val), np.random.randint(L_val)])

        neighbours = vonNeumanNeighborhood(i[0], i[1], L_val)
        spinSum = 0.
        for neighbour in neighbours:
            spinSum += current_lattice[neighbour[0], neighbour[1]]
        energy_difference = 2 * J * spinSum * current_lattice[i[0], i[1]]
        if energy_difference < 0:
            current_lattice[i[0], i[1]] *= (-1)
        else:
            r = np.random.uniform(0, 1)
            beta = 1. / T

            p = np.exp( (-1.) * beta * energy_difference )
            if r < p:
                current_lattice[i[0], i[1]] *= -1


magnetisationTable = dict()
magnetisationTableTheory = dict()
susceptibilityTable = dict()
corelationTable = []
t_c = 2.27

def iterate():
    global magnetisationTable, susceptibilityTable
    for L in Ls:
        avr_mag_T = []

        sus_T = []

        sim_lattice =  np.random.choice([-1, 1], size=(L, L))

        ms = np.array([])
        magnetisationForTemperature = np.array([])
        beta = 1. / T
        for i in range(N):
            mcs(T, L, sim_lattice)
            m = np.absolute( np.mean(sim_lattice) )
            ms = np.append(ms, m)

            magnetisation = np.abs( np.mean(sim_lattice) )

            magnetisationForTemperature = np.append(magnetisationForTemperature, magnetisation)

            if Ns.__contains__(i):
                print(sim_lattice)
                plt.imshow(sim_lattice, cmap='magma', interpolation='nearest')
                plt.title(f' {i:06d}')
                name = f'images/img{i:06d}.png'
                filenames.append(name)
                plt.savefig(name)
                plt.show()

                hi = []
                for r in r_0:
                    hi.append([r, hihi(sim_lattice, L, r)])

                hi = np.array(hi)
                plt.scatter(hi[:,0], hi[:,1], label="HIHI for N = {}".format(i))

                plt.title(f"HIHI")
                plt.xlabel("R_0")
                plt.ylabel("HIHI")
                plt.legend()
                plt.savefig(f"HIHI.png")




        print(magnetisationForTemperature.shape)
        averageMagnetisation = np.mean(magnetisationForTemperature)
        susceptibility = np.var(magnetisationForTemperature) * beta * L**2
        avr_mag_T.append([T, averageMagnetisation])
        sus_T.append([T, susceptibility])

        mag_average = np.mean(ms)
        magnetisationTable.update({L: np.array(avr_mag_T)})

        susceptibilityTable.update({L: np.array(sus_T)})

iterate()

for L in Ls:
    plt.scatter(magnetisationTable.get(L)[:,0], magnetisationTable.get(L)[:,1], label="Magnetisation for L = {}".format(L))

    plt.title(f"Magnetisation")
    plt.xlabel("Temperature")
    plt.ylabel("Magnetisation")
    plt.legend()
    plt.savefig(f"Magnetisation.png")

plt.show()

for L in Ls:
    plt.scatter(susceptibilityTable.get(L)[:,0], susceptibilityTable.get(L)[:,1], label="Susceptibility for L = {}".format(L))
    plt.title(f"Susceptibility")
    plt.xlabel("Temperature")
    plt.ylabel("Susceptibility")
    plt.legend()
    plt.savefig(f"Susceptibility.png")

plt.show()

with imageio.get_writer('images/movie.gif', mode='I') as writer:
    for filename in filenames:
        image = imageio.imread(filename)
        writer.append_data(image)



TypeError: only length-1 arrays can be converted to Python scalars